# Parte Comum — EDA (Wine Quality ou Obesity)

Projeto Final AVD 2026.2 — Grupo 6. Responsável: **Sacramento**.

Datasets permitidos pelo enunciado (o grupo usa **um**):
- [Wine Quality (UCI 186)](https://archive.ics.uci.edu/dataset/186): classe `quality`
- [Obesity (UCI 544)](https://archive.ics.uci.edu/dataset/544): classe `NObeyesdad`

**Antes de começar, leia `_CONTEXTO_IA/02_PARTE_COMUM.md`.**

## Decisão: qual dataset?

Wine Quality ou Obesity? Se for o Wine Quality: tinto, branco ou os dois?

Esta escolha define o número real de classes, que é o *k* da Passada 2 no `kmeans_metricas.ipynb`.

> **[Interpretação autoral — preencher]**

## Configuração

In [1]:
# [IA] Claude Code – célula de configuração (imports e caminhos do projeto)
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Sobe pastas até achar a raiz do repositório (onde está o requirements.txt),
# para o notebook funcionar não importa de onde o Jupyter foi aberto.
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "parte_comum"
OUT_DIR = ROOT / "out" / "parte_comum"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
print("Raiz do projeto encontrada:", ROOT.name)

Raiz do projeto encontrada: projeto-avd-2026-2-pump-it-up-classificatorio


## Carga dos dados

In [3]:
# [IA] Claude Code – leitura dos dois datasets da Parte Comum (o grupo usa um)
# Wine Quality: separador ";"
df_red = pd.read_csv(DATA_DIR / "wine_quality" / "winequality-red.csv", sep=";")
df_white = pd.read_csv(DATA_DIR / "wine_quality" / "winequality-white.csv", sep=";")
# Wine Quality com tinto e branco juntos (coluna wine_type = "red" ou "white")
df_wine_combined = pd.read_csv(DATA_DIR / "wine_quality" / "winequality-combined.csv", sep=";")
# Obesity: separador ","
df_obesity = pd.read_csv(DATA_DIR / "obesity" / "ObesityDataSet_raw_and_data_sinthetic.csv")
print("Wine tinto:", df_red.shape, "| Wine branco:", df_white.shape,
      "| Wine combinado:", df_wine_combined.shape, "| Obesity:", df_obesity.shape)

Wine tinto: (1599, 12) | Wine branco: (4898, 12) | Wine combinado: (6497, 13) | Obesity: (2111, 17)


In [4]:
# [IA] ChatGPT – selecionar o dataset Obesity e conferir a coluna de classe
df = df_obesity.copy()
coluna_classe = "NObeyesdad"

print("Quantidade de classes:", df[coluna_classe].nunique())

Quantidade de classes: 7


## 1. Análise Estrutural

Tipos de variáveis, dimensões, nulos, duplicados e estrutura geral.

In [5]:
# [IA] ChatGPT – conferir as dimensões, os tipos de dados e os valores preenchidos
print("Número de linhas:", df.shape[0])
print("Número de colunas:", df.shape[1])
df.info()

Número de linhas: 2111
Número de colunas: 17
<class 'pandas.DataFrame'>
RangeIndex: 2111 entries, 0 to 2110
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Gender                          2111 non-null   str    
 1   Age                             2111 non-null   float64
 2   Height                          2111 non-null   float64
 3   Weight                          2111 non-null   float64
 4   family_history_with_overweight  2111 non-null   str    
 5   FAVC                            2111 non-null   str    
 6   FCVC                            2111 non-null   float64
 7   NCP                             2111 non-null   float64
 8   CAEC                            2111 non-null   str    
 9   SMOKE                           2111 non-null   str    
 10  CH2O                            2111 non-null   float64
 11  SCC                             2111 non-null   str    
 12  

In [6]:
# [IA] ChatGPT – contar valores ausentes por coluna e linhas duplicadas
print("Valores ausentes por coluna:")
print(df.isna().sum())

print("Linhas duplicadas:", df.duplicated().sum())

Valores ausentes por coluna:
Gender                            0
Age                               0
Height                            0
Weight                            0
family_history_with_overweight    0
FAVC                              0
FCVC                              0
NCP                               0
CAEC                              0
SMOKE                             0
CH2O                              0
SCC                               0
FAF                               0
TUE                               0
CALC                              0
MTRANS                            0
NObeyesdad                        0
dtype: int64
Linhas duplicadas: 24


In [7]:
# [IA] ChatGPT – listar exemplos de registros idênticos e contar suas ocorrências
contagens = df.value_counts(dropna=False)
repetidos = contagens[contagens > 1]
repetidos.head(10).reset_index(name="ocorrencias")

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad,ocorrencias
0,Male,21.0,1.62,70.0,no,yes,2.0,1.0,no,no,3.0,no,1.0,0.0,Sometimes,Public_Transportation,Overweight_Level_I,15
1,Female,21.0,1.52,42.0,no,yes,3.0,1.0,Frequently,no,1.0,no,0.0,0.0,Sometimes,Public_Transportation,Insufficient_Weight,4
2,Female,21.0,1.52,42.0,no,no,3.0,1.0,Frequently,no,1.0,no,0.0,0.0,Sometimes,Public_Transportation,Insufficient_Weight,2
3,Female,25.0,1.57,55.0,no,yes,2.0,1.0,Sometimes,no,2.0,no,2.0,0.0,Sometimes,Public_Transportation,Normal_Weight,2
4,Female,22.0,1.69,65.0,yes,yes,2.0,3.0,Sometimes,no,2.0,no,1.0,1.0,Sometimes,Public_Transportation,Normal_Weight,2
5,Female,18.0,1.62,55.0,yes,yes,2.0,3.0,Frequently,no,1.0,no,1.0,1.0,no,Public_Transportation,Normal_Weight,2
6,Female,16.0,1.66,58.0,no,no,2.0,1.0,Sometimes,no,1.0,no,0.0,1.0,no,Walking,Normal_Weight,2
7,Male,18.0,1.72,53.0,yes,yes,2.0,3.0,Sometimes,no,2.0,no,0.0,2.0,Sometimes,Public_Transportation,Insufficient_Weight,2
8,Male,22.0,1.74,75.0,yes,yes,3.0,3.0,Frequently,no,1.0,no,1.0,0.0,no,Automobile,Normal_Weight,2


> **[Interpretação autoral — preencher]**

## 2. Análise Distribucional

Distribuição de cada variável, incluindo a classe.

In [ ]:
# TODO: histogramas/KDE/boxplots de cada variável e countplot da classe

> **[Interpretação autoral — preencher]**

## 3. Análise Relacional

Relações entre todas as variáveis, incluindo a variável de classe.

In [ ]:
# TODO: matriz de correlação (Pearson/Spearman) e relação de cada variável com a classe

> **[Interpretação autoral — preencher]**

## 4. Outliers

Identificação, discussão e tratamento (ou justificativa de não tratamento).

In [ ]:
# TODO: identificar outliers (IQR/boxplot ou z-score) e decidir o tratamento

> **[Interpretação autoral — preencher]**

## 5. Síntese da EDA

> **[Interpretação autoral — preencher]**